# TrustGuard: SMS Spam Model Experiment

This notebook is the guided, inspectable ML workflow for TrustGuard's first trained baseline. It covers data inspection, label balance, duplicate and conflicting-label analysis, training, model selection, held-out evaluation, error analysis, and sample predictions.

**Architecture note:** this is not the entire application in a notebook. Reusable production code stays in `src/trustguard/`, and repeatable command-line tasks stay in `scripts/`. This notebook calls the same training implementation used by the project scripts.

**Dataset:** UCI SMS Spam Collection, DOI `10.24432/C5CC84`; license listed by UCI as CC BY 4.0. This historical English SMS dataset is a narrow research benchmark, not a representative sample of every modern spam domain.

In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess

# Resolve the project root whether Jupyter starts in the repo root or in notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / "scripts" / "train_spam_classifier.py").exists():
    parent = ROOT.parent
    if (parent / "scripts" / "train_spam_classifier.py").exists():
        ROOT = parent
    else:
        raise RuntimeError("Could not find the TrustGuard project root. Start Jupyter from the repository or notebooks folder.")
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_PATH = ROOT / "data" / "raw" / "sms_spam.csv"
MODEL_PATH = ROOT / "models" / "spam_classifier.joblib"
REPORT_PATH = ROOT / "reports" / "spam_evaluation.json"
ERRORS_PATH = ROOT / "reports" / "spam_test_errors.csv"
print(f"Project root: {ROOT}")
print(f"Dataset exists: {DATA_PATH.exists()}")

## 1. Prepare the dataset

The dataset is deliberately excluded from Git. If it is not already present, the next cell runs the project's downloader. This requires internet access to the UCI dataset host. If your network blocks that host, follow the manual-download instructions in the repository README.

In [ ]:
if not DATA_PATH.exists():
    subprocess.run(
        [sys.executable, str(ROOT / "scripts" / "download_sms_dataset.py"), "--output", str(DATA_PATH)],
        cwd=ROOT,
        check=True,
    )

print(f"Dataset file: {DATA_PATH}")
print(f"File size: {DATA_PATH.stat().st_size:,} bytes")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_raw = pd.read_csv(DATA_PATH)
required = {"text", "label"}
if not required.issubset(df_raw.columns):
    raise ValueError(f"Expected CSV columns {required}; found {set(df_raw.columns)}")

df = df_raw[["text", "label"]].copy()
df["text"] = df["text"].fillna("").astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip().str.lower()
df = df[df["label"].isin(["ham", "spam"]) & df["text"].ne("")].reset_index(drop=True)
df["target"] = df["label"].map({"ham": 0, "spam": 1}).astype(int)

print(f"Raw rows loaded: {len(df_raw):,}")
print(f"Rows after basic validation: {len(df):,}")
display(df.head())

## 2. Inspect label balance and data quality

The exploration below is descriptive only. The training pipeline performs its own normalized-text deduplication and removes message groups with contradictory labels *before* creating the stratified train/validation/test split.

In [ ]:
label_counts = df["label"].value_counts().reindex(["ham", "spam"], fill_value=0)
label_summary = pd.DataFrame({
    "count": label_counts,
    "share": (label_counts / max(1, label_counts.sum())).round(4),
})
display(label_summary)

fig, ax = plt.subplots(figsize=(6, 4))
label_counts.plot(kind="bar", ax=ax)
ax.set_title("SMS Spam Collection: label distribution")
ax.set_xlabel("Label")
ax.set_ylabel("Number of messages")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
plt.show()

In [ ]:
# Normalize only for the duplicate audit; do not rewrite the original message content.
df["normalized_text"] = df["text"].str.casefold().str.split().str.join(" ")
rows = len(df)
unique_texts = df["normalized_text"].nunique()
duplicate_rows = rows - unique_texts
label_counts_by_text = df.groupby("normalized_text")["target"].nunique()
conflicting_groups = int((label_counts_by_text > 1).sum())

quality_summary = pd.DataFrame([
    {"measure": "Validated rows", "value": rows},
    {"measure": "Unique normalized message texts", "value": unique_texts},
    {"measure": "Repeated rows beyond one per normalized message", "value": duplicate_rows},
    {"measure": "Normalized messages with conflicting labels", "value": conflicting_groups},
])
display(quality_summary)

print("Example rows (fixed random seed):")
display(df[["text", "label"]].sample(min(8, len(df)), random_state=42))

## 3. Train and compare baselines

The shared training pipeline compares:

1. Word n-gram TF-IDF + Multinomial Naive Bayes.
2. Word and character TF-IDF + Logistic Regression.

It removes normalized duplicates and conflicting-label groups, creates a stratified 60/20/20 split, selects the model by validation macro-F1, chooses the operating threshold using validation data only, and reports the held-out test metrics once. Do not tune based on test performance.

In [ ]:
from trustguard.ml.training import train_and_save

texts = df["text"].tolist()
labels = df["target"].tolist()
report = train_and_save(
    texts,
    labels,
    task="spam",
    dataset_name="UCI SMS Spam Collection",
    dataset_source="https://archive.ics.uci.edu/dataset/228/sms+spam+collection (DOI: 10.24432/C5CC84)",
    label_rule="spam=1; ham=0",
    dataset_license="CC BY 4.0",
    output_dir=ROOT / "models",
    report_dir=ROOT / "reports",
    seed=42,
)

print("Selected model:", report["selection"]["selected_model"])
print("Saved model:", report["artifacts"]["model"])
print("Saved report:", REPORT_PATH)

In [ ]:
validation_rows = []
for model_name, values in report["selection"]["candidate_validation_results"].items():
    validation_rows.append({
        "model": model_name,
        "validation_macro_f1": values["validation_macro_f1"],
        "validation_precision_spam": values["validation_precision_positive"],
        "validation_recall_spam": values["validation_recall_positive"],
        "selected_threshold": values["threshold"],
        "precision_floor_met": bool(values["validation_selection"]["precision_floor_met"]),
    })
validation_table = pd.DataFrame(validation_rows).sort_values("validation_macro_f1", ascending=False)
display(validation_table)

## 4. Held-out test results

These values are generated by running the real-data pipeline. No benchmark values are pre-filled in this notebook. Model scores are not guaranteed to be calibrated probabilities, and the test set should not be used to keep tuning the model.

In [ ]:
test_metrics = report["test_metrics"]
summary_metrics = pd.DataFrame([
    {"metric": "Accuracy", "value": test_metrics["accuracy"]},
    {"metric": "Spam precision", "value": test_metrics["precision_positive"]},
    {"metric": "Spam recall", "value": test_metrics["recall_positive"]},
    {"metric": "Spam F1", "value": test_metrics["f1_positive"]},
    {"metric": "Macro F1", "value": test_metrics["macro_f1"]},
    {"metric": "False-positive rate", "value": test_metrics["false_positive_rate"]},
    {"metric": "Decision threshold", "value": test_metrics["threshold"]},
])
display(summary_metrics)
print("Test set size:", test_metrics["n_test"])

In [ ]:
cm = np.asarray(test_metrics["confusion_matrix_labels_0_1"], dtype=int)
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(cm)
ax.set_title("Held-out test confusion matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_xticks([0, 1], labels=["Ham", "Spam"])
ax.set_yticks([0, 1], labels=["Ham", "Spam"])
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, str(value), ha="center", va="center")
fig.colorbar(image, ax=ax, label="Messages")
fig.tight_layout()
plt.show()

print("TN, FP, FN, TP:", test_metrics["counts"])

In [ ]:
class_report = test_metrics["classification_report"]
per_class = pd.DataFrame(class_report).T
per_class = per_class.loc[[name for name in ["negative", "positive", "macro avg", "weighted avg"] if name in per_class.index]]
per_class = per_class.rename(index={"negative": "Ham (negative)", "positive": "Spam (positive)"})
display(per_class.round(4))

## 5. Error analysis

Misclassified examples are written to `reports/spam_test_errors.csv`. This file contains full message text; keep it local and do not commit it by default. Inspect a small sample to understand the failure modes rather than only reporting a single metric.

In [ ]:
if ERRORS_PATH.exists():
    errors = pd.read_csv(ERRORS_PATH)
    print(f"Misclassified test examples: {len(errors):,}")
    if errors.empty:
        print("No errors at the selected threshold in this test split.")
    else:
        display(errors.head(20))
else:
    print("Error export not found. Run the training cell above first.")

## 6. Try a few messages against the saved model

These are illustrative smoke tests, not part of the benchmark. Do not use a few hand-picked messages to claim model quality.

In [ ]:
import joblib

artifact = joblib.load(MODEL_PATH)
estimator = artifact["estimator"]
threshold = float(artifact["threshold"])
positive_index = list(estimator.classes_).index(1)

def score_message(message: str) -> dict:
    score = float(estimator.predict_proba([message])[0, positive_index])
    return {
        "message": message,
        "spam_model_score": round(score, 4),
        "selected_threshold": round(threshold, 4),
        "predicted_label": "spam" if score >= threshold else "ham",
    }

samples = [
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT! You have won a prize. Claim now at https://example.invalid",
    "Please review the meeting notes when you have time.",
    "Congratulations!!! Call now to claim your FREE reward!!!",
]
display(pd.DataFrame([score_message(message) for message in samples]))

## 7. Interpretation and limitations

Before using this model as one signal inside TrustGuard:

- Review the confusion matrix, false positives, false negatives, and error examples.
- Do not treat the classifier score as a calibrated probability.
- SMS spam differs from spam in forums, reviews, social media, and email.
- Language, demographics, writing style, and changing adversarial tactics can affect error rates.
- The model must not independently ban users or permanently delete content. Combine it with policy checks, context, uncertainty handling, and human review.
- Preserve data attribution, and do not add raw data, local test-error exports, secrets, or unreviewed model artifacts to Git.

The generated `reports/spam_evaluation.json` and `reports/spam_model_card.md` are the machine-readable and human-readable records of this run.